In [1]:
!pip install scipy


# ============================================================
# Q4 ATTENTION BENCHMARK — BURIED CRITICAL INFORMATION
# ------------------------------------------------------------
# Question:
# How does the model perform when critical information is buried among large
# amounts of irrelevant context?
#
# Core design:
# - Keep the target article fixed.
# - Keep total irrelevant context fixed across conditions.
# - Vary ONLY where the target article is placed in a long dossier.
#
# Conditions:
# 1) target_early
# 2) target_middle
# 3) target_late
#
# Why this is stronger than reusing the Q2 length probe:
# - Q2 changes total irrelevant context length.
# - Q4 here keeps total dossier length roughly fixed and changes only
#   target position / buriedness.
# - This isolates retrieval and sustained attention to buried relevant content.
# ============================================================

import hashlib
import json
import math
import re
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import binomtest
from tqdm.auto import tqdm

import kaggle_benchmarks as kbench

# ---------------------------
# CONFIG
# ---------------------------
INPUT_CSV = "/kaggle/input/datasets/bigdataanalysis1/epu-800/EPU_benchmark_800_balanced.csv"

SEED = 42
N_ROWS = 600            # pilot size; raise to 600 when stable
MAX_CHARS = 4500
BOOT_N = 5000
N_DISTRACTORS = 5       # fixed across all Q4 conditions

MODEL_NAMES = [
    # "google/gemini-2.5-flash",
    # "google/gemini-2.5-pro",
    # "anthropic/claude-sonnet-4-6@default",
    # "deepseek-ai/deepseek-v3.2",
    "qwen/qwen3-235b-a22b-instruct-2507"
]

OUT_DIR = Path("/kaggle/working/epu_attention_q4_buried_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MISSING_SENTINEL = -1

# ---------------------------
# LOAD + CLEAN
# ---------------------------
df = pd.read_csv(INPUT_CSV)

required_cols = [
    "article_key",
    "content",
    "gold_epu",
    "gold_who",
    "gold_actions",
    "gold_effects",
    "mention_foreign",
    "mainly_foreign",
    "disagreement_flag",
    "newspaper",
    "year",
    "content_len",
]
for c in required_cols:
    if c not in df.columns:
        df[c] = np.nan

mask = (
    df["content"].notna()
    & (df["content"].astype(str).str.len() > 300)
    & df["gold_epu"].notna()
)

clean = df.loc[mask].copy()
clean["article_key"] = clean["article_key"].astype(str)
missing_key_mask = clean["article_key"].isin(["nan", "None", ""])
if missing_key_mask.any():
    clean.loc[missing_key_mask, "article_key"] = [
        f"generated_key_{i}" for i in range(missing_key_mask.sum())
    ]

clean["gold_epu"] = clean["gold_epu"].astype(int)
clean["gold_mention_foreign"] = (clean["mention_foreign"].fillna(0) >= 0.5).astype(int)
clean["gold_mainly_foreign"] = (clean["mainly_foreign"].fillna(0) >= 0.5).astype(int)
clean["gold_who"] = clean["gold_who"].fillna(MISSING_SENTINEL).astype(int)
clean["gold_actions"] = clean["gold_actions"].fillna(MISSING_SENTINEL).astype(int)
clean["gold_effects"] = clean["gold_effects"].fillna(MISSING_SENTINEL).astype(int)
clean["disagreement_flag"] = clean["disagreement_flag"].fillna(0).astype(int)
clean = clean.drop_duplicates(subset=["article_key"]).reset_index(drop=True)

print(f"Loaded rows: {len(df):,}")
print(f"Clean usable rows: {len(clean):,}")
print(clean["gold_epu"].value_counts().sort_index())
print(clean["disagreement_flag"].value_counts().sort_index())

# ---------------------------
# BALANCED SAMPLE
# ---------------------------
def balanced_sample(data: pd.DataFrame, n: int, seed: int = 42) -> pd.DataFrame:
    pos = data[data["gold_epu"] == 1].copy()
    neg = data[data["gold_epu"] == 0].copy()

    if len(pos) == 0 or len(neg) == 0:
        return data.sample(min(n, len(data)), random_state=seed).reset_index(drop=True)

    n_pos = min(len(pos), n // 2)
    n_neg = min(len(neg), n - n_pos)

    pos_s = pos.sample(n=n_pos, random_state=seed)
    neg_s = neg.sample(n=n_neg, random_state=seed + 1)

    return (
        pd.concat([pos_s, neg_s], axis=0)
        .sample(frac=1, random_state=seed + 2)
        .reset_index(drop=True)
    )

pilot = balanced_sample(clean, N_ROWS, SEED).copy()

# ---------------------------
# FIXED-LENGTH DISTRACTOR DOSSIER
# ---------------------------
DISTRACTORS = [
    "A politics brief discusses campaign speeches, polling swings, and coalition rumors. It sounds policy-adjacent but does not establish economic policy uncertainty in the article to classify.",
    "A market recap describes stock volatility, earnings surprises, and cautious investors. It is economically salient but not itself evidence about government policy uncertainty in the target article.",
    "A foreign affairs report covers diplomatic tension, troop movement, and sanctions debate abroad. It includes uncertainty language but is not the article of interest.",
    "A weather and transport bulletin details storms, airport delays, and supply interruptions. It is vivid but irrelevant to the target article's EPU coding.",
    "A sports story covers injuries, coaching changes, and playoff pressure. It contains uncertainty words but no economic policy uncertainty for the target article.",
    "An entertainment report discusses streaming contracts, celebrity disputes, and release delays. It is attention-grabbing but irrelevant to the target article.",
    "A science note summarizes a disputed medical study and cautious expert reactions. It contains uncertainty but not the target article's policy content.",
    "A city feature covers retail traffic, tourism, local events, and downtown parking issues. It may mention business conditions but not the target article.",
    "A housing note describes home sales, mortgage chatter, and buyer hesitation. It sounds economic, but it is not the section to classify.",
    "An education story covers testing changes, district plans, and parent complaints. It is not the article of interest for the benchmark task.",
]

SECTION_IDS = ["A17", "B29", "C41", "D53", "E67", "F79"]

def stable_index(key: str, mod: int) -> int:
    h = hashlib.md5(str(key).encode("utf-8")).hexdigest()
    return int(h[:8], 16) % mod

def pick_distractor(article_key: str, offset: int = 0) -> str:
    idx = stable_index(f"{article_key}_q4_{offset}", len(DISTRACTORS))
    return DISTRACTORS[idx]

def smart_truncate(text: str, max_chars: int = 4500) -> str:
    text = str(text)
    if len(text) <= max_chars:
        return text
    head = max_chars // 2
    tail = max_chars - head - 32
    return text[:head] + "\n\n[...TRUNCATED...]\n\n" + text[-tail:]

pilot["content_for_eval"] = pilot["content"].astype(str).apply(lambda x: smart_truncate(x, MAX_CHARS))

def build_dossier(article_key: str, target_text: str, target_position: str):
    # 6 total sections = 5 distractors + 1 target article
    distractor_texts = [pick_distractor(article_key, i) for i in range(N_DISTRACTORS)]
    distractor_sections = [f"SECTION ID {sid}\n{text}" for sid, text in zip(SECTION_IDS[:N_DISTRACTORS], distractor_texts)]

    target_section_id = SECTION_IDS[-1]
    target_section_text = f"SECTION ID {target_section_id}\n{target_text}"

    if target_position == "target_early":
        insert_at = 1   # 2nd section
    elif target_position == "target_middle":
        insert_at = 3   # 4th section
    elif target_position == "target_late":
        insert_at = 5   # 6th / final section
    else:
        raise ValueError(f"Unknown target_position: {target_position}")

    sections = distractor_sections.copy()
    sections.insert(insert_at, target_section_text)

    dossier_text = "\n\n-----\n\n".join(sections)
    return {
        "target_section_id": target_section_id,
        "dossier_text": dossier_text,
        "target_position": target_position,
        "insert_at_index": insert_at + 1,  # human-readable 1-based position
    }

for cond in ["target_early", "target_middle", "target_late"]:
    built = pilot.apply(lambda r: build_dossier(r["article_key"], r["content_for_eval"], cond), axis=1)
    pilot[f"{cond}_target_section_id"] = built.apply(lambda x: x["target_section_id"])
    pilot[f"{cond}_dossier_text"] = built.apply(lambda x: x["dossier_text"])
    pilot[f"{cond}_section_position"] = built.apply(lambda x: x["insert_at_index"])

pilot_path = OUT_DIR / "pilot_rows_q4.csv"
pilot.to_csv(pilot_path, index=False)

# ---------------------------
# HELPERS
# ---------------------------
def maybe_int(x):
    if x is None:
        return None
    try:
        if pd.isna(x):
            return None
    except Exception:
        pass
    try:
        return int(x)
    except Exception:
        return None

def binary_match(pred, gold, missing_sentinel=MISSING_SENTINEL):
    gold_i = maybe_int(gold)
    pred_i = maybe_int(pred)
    if gold_i is None or gold_i == missing_sentinel:
        return None
    if pred_i is None:
        return 0.0
    return 1.0 if pred_i == gold_i else 0.0

def safe_float(x, default=np.nan):
    try:
        return float(x)
    except Exception:
        return default

def default_out():
    return {
        "epu": None,
        "confidence": 0.0,
        "focus_excerpt": "",
        "raw_text": "",
        "json_valid": 0,
    }

def parse_llm_json(raw_text):
    out = default_out()
    out["raw_text"] = "" if raw_text is None else str(raw_text)[:4000]

    if raw_text is None:
        return out

    text = str(raw_text).strip()
    text = text.replace("```json", "").replace("```", "").strip()

    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match:
        text = match.group(0)

    try:
        obj = json.loads(text)
        if not isinstance(obj, dict):
            return out
    except Exception:
        return out

    out["json_valid"] = 1

    for k in ["epu", "confidence", "focus_excerpt"]:
        if k in obj:
            out[k] = obj[k]

    out["epu"] = maybe_int(out["epu"])

    try:
        out["confidence"] = float(out["confidence"])
    except Exception:
        out["confidence"] = 0.0

    out["confidence"] = max(0.0, min(1.0, out["confidence"]))
    out["focus_excerpt"] = "" if out["focus_excerpt"] is None else str(out["focus_excerpt"])[:250]
    return out

def model_json_call(llm, prompt: str):
    try:
        raw = llm.prompt(prompt)
        return parse_llm_json(raw)
    except Exception as e:
        out = default_out()
        out["raw_text"] = f"ERROR: {str(e)[:4000]}"
        return out

def excerpt_in_target(excerpt: str, article_text: str) -> int:
    excerpt = "" if excerpt is None else str(excerpt).strip()
    article_text = "" if article_text is None else str(article_text)
    if len(excerpt) < 12:
        return 0
    return int(excerpt in article_text)

# ---------------------------
# PROMPT
# ---------------------------
def build_buried_prompt(target_section_id: str, dossier_text: str) -> str:
    return (
        "You are given a dossier with multiple sectioned articles.\n\n"
        f"Evaluate ONLY the article in SECTION ID {target_section_id}.\n"
        "Ignore every other section, even if it contains politics, markets, war, or uncertainty language.\n\n"
        "Task:\n"
        "Decide whether SECTION ID "
        f"{target_section_id} "
        "expresses ECONOMIC POLICY UNCERTAINTY (EPU).\n\n"
        "Labeling rule:\n"
        "- EPU = 1 only if the target section discusses uncertainty about government policy,\n"
        "  policy decisions, possible policy actions, political outcomes affecting policy,\n"
        "  or the economic effects of policy.\n"
        "- EPU = 0 for general economics, elections, markets, war, uncertainty, or foreign events\n"
        "  unless the uncertainty is specifically tied to economic policy.\n\n"
        "IMPORTANT:\n"
        "- The relevant section may appear anywhere in the dossier.\n"
        "- Use ONLY the named section ID.\n"
        "- Return ONLY valid JSON.\n"
        "- No markdown. No explanation. No code fence.\n\n"
        "Return exactly these keys:\n"
        "{\n"
        '  "epu": 0 or 1,\n'
        '  "confidence": number between 0 and 1,\n'
        '  "focus_excerpt": "short quote from the target section only"\n'
        "}\n\n"
        "DOSSIER START\n"
        f"{dossier_text}\n"
        "DOSSIER END"
    ).strip()

# ---------------------------
# STATISTICS
# ---------------------------
def wilson_ci(successes: int, n: int, z: float = 1.959963984540054):
    if n <= 0:
        return (np.nan, np.nan)
    phat = successes / n
    denom = 1.0 + (z**2) / n
    center = (phat + (z**2) / (2 * n)) / denom
    margin = (z / denom) * math.sqrt((phat * (1 - phat) / n) + (z**2) / (4 * n**2))
    return (max(0.0, center - margin), min(1.0, center + margin))

def paired_bootstrap_diff_ci(a_values, b_values, n_boot=5000, seed=42, alpha=0.05):
    a = np.asarray(a_values, dtype=float)
    b = np.asarray(b_values, dtype=float)
    mask = ~(np.isnan(a) | np.isnan(b))
    a = a[mask]
    b = b[mask]
    n = len(a)
    if n == 0:
        return (np.nan, np.nan)
    diffs = a - b
    rng = np.random.default_rng(seed)
    boot = np.empty(n_boot, dtype=float)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boot[i] = diffs[idx].mean()
    return (float(np.quantile(boot, alpha / 2)), float(np.quantile(boot, 1 - alpha / 2)))

def exact_mcnemar(a_correct, b_correct):
    x = np.asarray(a_correct, dtype=float)
    y = np.asarray(b_correct, dtype=float)
    mask = ~(np.isnan(x) | np.isnan(y))
    x = x[mask].astype(int)
    y = y[mask].astype(int)
    b = int(np.sum((x == 1) & (y == 0)))
    c = int(np.sum((x == 0) & (y == 1)))
    n = b + c
    if n == 0:
        return {
            "b_hurt": b,
            "c_helped": c,
            "discordant_n": n,
            "mcnemar_p": 1.0,
            "hurt_help_ratio": np.nan,
        }
    pval = binomtest(min(b, c), n=n, p=0.5, alternative="two-sided").pvalue
    ratio = np.inf if c == 0 and b > 0 else (b / c if c > 0 else np.nan)
    return {
        "b_hurt": b,
        "c_helped": c,
        "discordant_n": n,
        "mcnemar_p": float(pval),
        "hurt_help_ratio": ratio,
    }

def classify_buried(drop_pp, flip_rate, pval):
    if pd.isna(drop_pp) or pd.isna(flip_rate):
        return "undetermined"
    if drop_pp >= 0.10 and flip_rate >= 0.15 and pval < 0.05:
        return "strong buried-information failure"
    if drop_pp >= 0.03 and flip_rate >= 0.05 and pval < 0.05:
        return "moderate buried-information failure"
    if drop_pp < 0.03 and flip_rate < 0.05:
        return "little buried-information failure"
    return "mixed / borderline"

def summarize_pair(a_correct, b_correct, a_pred, b_pred, compare_name):
    a = np.asarray(a_correct, dtype=float)
    b = np.asarray(b_correct, dtype=float)
    pred_a = np.asarray(a_pred, dtype=float)
    pred_b = np.asarray(b_pred, dtype=float)

    mask = ~(np.isnan(a) | np.isnan(b))
    a = a[mask]
    b = b[mask]
    pred_a = pred_a[mask]
    pred_b = pred_b[mask]

    n = len(a)
    acc_a = float(np.mean(a)) if n else np.nan
    acc_b = float(np.mean(b)) if n else np.nan
    drop_pp = acc_a - acc_b if n else np.nan
    drop_lo, drop_hi = paired_bootstrap_diff_ci(a, b, n_boot=BOOT_N, seed=SEED)

    flip_mask = ~(np.isnan(pred_a) | np.isnan(pred_b))
    flip_n = int(np.sum(flip_mask))
    flip_k = int(np.sum(pred_a[flip_mask] != pred_b[flip_mask])) if flip_n else 0
    flip_rate = flip_k / flip_n if flip_n else np.nan
    flip_lo, flip_hi = wilson_ci(flip_k, flip_n)

    hurt_k = int(np.sum((a == 1) & (b == 0))) if n else 0
    help_k = int(np.sum((a == 0) & (b == 1))) if n else 0
    mc = exact_mcnemar(a, b)

    return {
        "comparison": compare_name,
        "n_rows": n,
        "acc_a": acc_a,
        "acc_b": acc_b,
        "drop_pp": drop_pp,
        "drop_ci_low": drop_lo,
        "drop_ci_high": drop_hi,
        "flip_rate": flip_rate,
        "flip_rate_ci_low": flip_lo,
        "flip_rate_ci_high": flip_hi,
        "hurt_rate": hurt_k / n if n else np.nan,
        "help_rate": help_k / n if n else np.nan,
        **mc,
        "buried_failure_level": classify_buried(drop_pp, flip_rate, mc["mcnemar_p"]),
    }

# ---------------------------
# RUN MODELS
# ---------------------------
conditions = ["target_early", "target_middle", "target_late"]
records = []
model_handles = {name: kbench.llms[name] for name in MODEL_NAMES}

for model_name in MODEL_NAMES:
    llm = model_handles[model_name]
    print(f"\nRunning {model_name} ...")

    for row in tqdm(pilot.to_dict(orient="records"), total=len(pilot), desc=model_name):
        rec = {
            "llm_name": model_name,
            "article_key": row["article_key"],
            "gold_epu": row["gold_epu"],
            "gold_who": row["gold_who"],
            "gold_actions": row["gold_actions"],
            "gold_effects": row["gold_effects"],
            "gold_mention_foreign": row["gold_mention_foreign"],
            "gold_mainly_foreign": row["gold_mainly_foreign"],
            "disagreement_flag": row["disagreement_flag"],
            "newspaper": row.get("newspaper", ""),
            "year": row.get("year", np.nan),
            "content_len": row.get("content_len", np.nan),
            "content_for_eval": row["content_for_eval"],
        }

        for cond in conditions:
            target_section_id = row[f"{cond}_target_section_id"]
            dossier_text = row[f"{cond}_dossier_text"]
            section_position = row[f"{cond}_section_position"]

            out = model_json_call(llm, build_buried_prompt(target_section_id, dossier_text))

            rec[f"{cond}_target_section_id"] = target_section_id
            rec[f"{cond}_section_position"] = section_position
            rec[f"{cond}_dossier_text"] = dossier_text
            rec[f"{cond}_epu_pred"] = maybe_int(out["epu"])
            rec[f"{cond}_epu_correct"] = binary_match(out["epu"], row["gold_epu"])
            rec[f"{cond}_confidence"] = safe_float(out["confidence"])
            rec[f"{cond}_focus_excerpt"] = out["focus_excerpt"]
            rec[f"{cond}_excerpt_in_target"] = excerpt_in_target(out["focus_excerpt"], row["content_for_eval"])
            rec[f"{cond}_json_valid"] = int(out["json_valid"])
            rec[f"{cond}_raw_json"] = out["raw_text"]

        rec["early_to_late_hurt"] = int(rec["target_early_epu_correct"] == 1.0 and rec["target_late_epu_correct"] == 0.0) if not pd.isna(rec["target_early_epu_correct"]) and not pd.isna(rec["target_late_epu_correct"]) else 0
        records.append(rec)

wide_df = pd.DataFrame(records)
wide_path = OUT_DIR / "q4_results_wide.csv"
wide_df.to_csv(wide_path, index=False)

# ---------------------------
# LONG VERSION
# ---------------------------
long_records = []
for row in wide_df.to_dict(orient="records"):
    base = {
        "llm_name": row["llm_name"],
        "article_key": row["article_key"],
        "gold_epu": row["gold_epu"],
        "disagreement_flag": row["disagreement_flag"],
        "newspaper": row["newspaper"],
        "year": row["year"],
        "content_len": row["content_len"],
    }
    for cond in conditions:
        rec = dict(base)
        rec["condition"] = cond
        rec["target_section_id"] = row[f"{cond}_target_section_id"]
        rec["section_position"] = row[f"{cond}_section_position"]
        rec["epu_pred"] = row[f"{cond}_epu_pred"]
        rec["epu_correct"] = row[f"{cond}_epu_correct"]
        rec["confidence"] = row[f"{cond}_confidence"]
        rec["focus_excerpt"] = row[f"{cond}_focus_excerpt"]
        rec["excerpt_in_target"] = row[f"{cond}_excerpt_in_target"]
        rec["json_valid"] = row[f"{cond}_json_valid"]
        long_records.append(rec)

long_df = pd.DataFrame(long_records)
long_path = OUT_DIR / "q4_results_long.csv"
long_df.to_csv(long_path, index=False)

# ---------------------------
# SUMMARY TABLES
# ---------------------------
summary_rows = []
for model_name, g in wide_df.groupby("llm_name", sort=False):
    rec = {"llm_name": model_name, "n_rows": len(g)}

    for cond in conditions:
        vec = g[f"{cond}_epu_correct"].astype(float).to_numpy()
        n = int(np.sum(~np.isnan(vec)))
        k = int(np.nansum(vec)) if n > 0 else 0
        mean_acc = float(np.nanmean(vec)) if n > 0 else np.nan
        lo, hi = wilson_ci(k, n)

        rec[f"{cond}_acc"] = mean_acc
        rec[f"{cond}_acc_ci_low"] = lo
        rec[f"{cond}_acc_ci_high"] = hi
        rec[f"{cond}_json_valid_rate"] = g[f"{cond}_json_valid"].mean()
        rec[f"{cond}_excerpt_in_target_rate"] = g[f"{cond}_excerpt_in_target"].mean()
        rec[f"{cond}_confidence_mean"] = g[f"{cond}_confidence"].mean()

    comp_early_middle = summarize_pair(
        g["target_early_epu_correct"],
        g["target_middle_epu_correct"],
        g["target_early_epu_pred"],
        g["target_middle_epu_pred"],
        "target_early -> target_middle",
    )
    comp_early_late = summarize_pair(
        g["target_early_epu_correct"],
        g["target_late_epu_correct"],
        g["target_early_epu_pred"],
        g["target_late_epu_pred"],
        "target_early -> target_late",
    )
    comp_middle_late = summarize_pair(
        g["target_middle_epu_correct"],
        g["target_late_epu_correct"],
        g["target_middle_epu_pred"],
        g["target_late_epu_pred"],
        "target_middle -> target_late",
    )

    rec["early_to_middle_drop_pp"] = comp_early_middle["drop_pp"]
    rec["early_to_middle_mcnemar_p"] = comp_early_middle["mcnemar_p"]
    rec["early_to_middle_flip_rate"] = comp_early_middle["flip_rate"]

    rec["early_to_late_drop_pp"] = comp_early_late["drop_pp"]
    rec["early_to_late_drop_ci_low"] = comp_early_late["drop_ci_low"]
    rec["early_to_late_drop_ci_high"] = comp_early_late["drop_ci_high"]
    rec["early_to_late_mcnemar_p"] = comp_early_late["mcnemar_p"]
    rec["early_to_late_flip_rate"] = comp_early_late["flip_rate"]
    rec["early_to_late_buried_failure_level"] = comp_early_late["buried_failure_level"]

    rec["middle_to_late_drop_pp"] = comp_middle_late["drop_pp"]
    rec["middle_to_late_mcnemar_p"] = comp_middle_late["mcnemar_p"]
    rec["middle_to_late_flip_rate"] = comp_middle_late["flip_rate"]

    summary_rows.append(rec)

summary_overall = pd.DataFrame(summary_rows).sort_values("early_to_late_drop_pp", ascending=False)
summary_overall_path = OUT_DIR / "q4_summary_overall.csv"
summary_overall.to_csv(summary_overall_path, index=False)

pairwise_rows = []
for model_name, g in wide_df.groupby("llm_name", sort=False):
    pairwise_rows.append(dict({"llm_name": model_name}, **summarize_pair(
        g["target_early_epu_correct"],
        g["target_middle_epu_correct"],
        g["target_early_epu_pred"],
        g["target_middle_epu_pred"],
        "target_early -> target_middle",
    )))
    pairwise_rows.append(dict({"llm_name": model_name}, **summarize_pair(
        g["target_early_epu_correct"],
        g["target_late_epu_correct"],
        g["target_early_epu_pred"],
        g["target_late_epu_pred"],
        "target_early -> target_late",
    )))
    pairwise_rows.append(dict({"llm_name": model_name}, **summarize_pair(
        g["target_middle_epu_correct"],
        g["target_late_epu_correct"],
        g["target_middle_epu_pred"],
        g["target_late_epu_pred"],
        "target_middle -> target_late",
    )))

pairwise_df = pd.DataFrame(pairwise_rows)
pairwise_path = OUT_DIR / "q4_pairwise_proof.csv"
pairwise_df.to_csv(pairwise_path, index=False)

# ---------------------------
# BY DISAGREEMENT / BY GOLD LABEL
# ---------------------------
def summarize_wide_group(g: pd.DataFrame):
    rec = {"n_rows": len(g)}
    for cond in conditions:
        rec[f"{cond}_acc"] = g[f"{cond}_epu_correct"].mean()
        rec[f"{cond}_json_valid_rate"] = g[f"{cond}_json_valid"].mean()
        rec[f"{cond}_excerpt_in_target_rate"] = g[f"{cond}_excerpt_in_target"].mean()
    comp = summarize_pair(
        g["target_early_epu_correct"],
        g["target_late_epu_correct"],
        g["target_early_epu_pred"],
        g["target_late_epu_pred"],
        "target_early -> target_late",
    )
    rec["early_to_late_drop_pp"] = comp["drop_pp"]
    rec["early_to_late_mcnemar_p"] = comp["mcnemar_p"]
    rec["early_to_late_flip_rate"] = comp["flip_rate"]
    rec["early_to_late_buried_failure_level"] = comp["buried_failure_level"]
    return rec

by_disagreement_rows = []
for (model_name, flag), g in wide_df.groupby(["llm_name", "disagreement_flag"], sort=False):
    by_disagreement_rows.append(
        dict({"llm_name": model_name, "disagreement_flag": int(flag)}, **summarize_wide_group(g))
    )

by_disagreement_df = pd.DataFrame(by_disagreement_rows)
by_disagreement_path = OUT_DIR / "q4_summary_by_disagreement.csv"
by_disagreement_df.to_csv(by_disagreement_path, index=False)

by_gold_rows = []
for (model_name, gold_epu), g in wide_df.groupby(["llm_name", "gold_epu"], sort=False):
    by_gold_rows.append(
        dict({"llm_name": model_name, "gold_epu": int(gold_epu)}, **summarize_wide_group(g))
    )

by_gold_df = pd.DataFrame(by_gold_rows)
by_gold_path = OUT_DIR / "q4_summary_by_gold_epu.csv"
by_gold_df.to_csv(by_gold_path, index=False)

# ---------------------------
# RETRIEVAL / HARDEST ROWS
# ---------------------------
retrieval_rows = []
for model_name, g in wide_df.groupby("llm_name", sort=False):
    for cond in conditions:
        retrieval_rows.append({
            "llm_name": model_name,
            "condition": cond,
            "json_valid_rate": g[f"{cond}_json_valid"].mean(),
            "excerpt_in_target_rate": g[f"{cond}_excerpt_in_target"].mean(),
            "accuracy": g[f"{cond}_epu_correct"].mean(),
            "confidence_mean": g[f"{cond}_confidence"].mean(),
        })

retrieval_df = pd.DataFrame(retrieval_rows)
retrieval_path = OUT_DIR / "q4_retrieval_table.csv"
retrieval_df.to_csv(retrieval_path, index=False)

harmed_rows = (
    wide_df.groupby(["article_key", "gold_epu", "disagreement_flag"], as_index=False)
    .agg(
        harmed_models_early_to_late=("early_to_late_hurt", "sum"),
        mean_early_acc=("target_early_epu_correct", "mean"),
        mean_middle_acc=("target_middle_epu_correct", "mean"),
        mean_late_acc=("target_late_epu_correct", "mean"),
        content_for_eval=("content_for_eval", "first"),
        target_early_target_section_id=("target_early_target_section_id", "first"),
    )
    .sort_values(["harmed_models_early_to_late", "mean_late_acc"], ascending=[False, True])
)
harmed_path = OUT_DIR / "q4_most_harmed_rows.csv"
harmed_rows.to_csv(harmed_path, index=False)

# ---------------------------
# MANIFEST
# ---------------------------
manifest = pd.DataFrame({
    "file_name": [
        wide_path.name,
        long_path.name,
        summary_overall_path.name,
        pairwise_path.name,
        by_disagreement_path.name,
        by_gold_path.name,
        retrieval_path.name,
        harmed_path.name,
        pilot_path.name,
    ],
    "description": [
        "Row-level wide table with one row per model x article and predictions for early, middle, and late target positions.",
        "Long-format table with one row per model x article x condition.",
        "Main Q4 summary table with condition accuracies and the strongest buried-information effect.",
        "Paired scientific proof for early->middle, early->late, and middle->late comparisons.",
        "Q4 metrics split by disagreement_flag.",
        "Q4 metrics split by gold_epu.",
        "Support table for retrieval quality: excerpt-in-target rate, JSON validity, accuracy, and confidence.",
        "Rows most harmed when the target article is buried late rather than early.",
        "The exact pilot rows used for Q4.",
    ]
})
manifest_path = OUT_DIR / "q4_manifest.csv"
manifest.to_csv(manifest_path, index=False)

print("\nSaved files:")
for p in [
    wide_path, long_path, summary_overall_path, pairwise_path, by_disagreement_path,
    by_gold_path, retrieval_path, harmed_path, pilot_path, manifest_path
]:
    print(p)

print("\nQ4 overall summary:")
display(summary_overall)

print("\nQ4 pairwise proof:")
display(pairwise_df)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/35.3 MB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 35.1/35.3 MB 284.3 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 130.1 MB/s  0:00:00


Loaded rows: 800
Clean usable rows: 800
gold_epu
0    400
1    400
Name: count, dtype: int64
disagreement_flag
0    585
1    215
Name: count, dtype: int64



Running qwen/qwen3-235b-a22b-instruct-2507 ...


qwen/qwen3-235b-a22b-instruct-2507:   0%|          | 0/600 [00:00<?, ?it/s]


Saved files:
/kaggle/working/epu_attention_q4_buried_outputs/q4_results_wide.csv
/kaggle/working/epu_attention_q4_buried_outputs/q4_results_long.csv
/kaggle/working/epu_attention_q4_buried_outputs/q4_summary_overall.csv
/kaggle/working/epu_attention_q4_buried_outputs/q4_pairwise_proof.csv
/kaggle/working/epu_attention_q4_buried_outputs/q4_summary_by_disagreement.csv
/kaggle/working/epu_attention_q4_buried_outputs/q4_summary_by_gold_epu.csv
/kaggle/working/epu_attention_q4_buried_outputs/q4_retrieval_table.csv
/kaggle/working/epu_attention_q4_buried_outputs/q4_most_harmed_rows.csv
/kaggle/working/epu_attention_q4_buried_outputs/pilot_rows_q4.csv
/kaggle/working/epu_attention_q4_buried_outputs/q4_manifest.csv

Q4 overall summary:


,llm_name,n_rows,target_early_acc,target_early_acc_ci_low,target_early_acc_ci_high,target_early_json_valid_rate,target_early_excerpt_in_target_rate,target_early_confidence_mean,target_middle_acc,target_middle_acc_ci_low,...,early_to_middle_flip_rate,early_to_late_drop_pp,early_to_late_drop_ci_low,early_to_late_drop_ci_high,early_to_late_mcnemar_p,early_to_late_flip_rate,early_to_late_buried_failure_level,middle_to_late_drop_pp,middle_to_late_mcnemar_p,middle_to_late_flip_rate
0,qwen/qwen3-235b-a22b-instruct-2507,600,0.65,0.610991,0.687101,0.988333,0.648333,0.523083,0.663333,0.624588,...,0.071066,-0.008333,-0.033333,0.016667,0.596642,0.088136,mixed / borderline,0.005,0.779768,0.084175



Q4 pairwise proof:


,llm_name,comparison,n_rows,acc_a,acc_b,drop_pp,drop_ci_low,drop_ci_high,flip_rate,flip_rate_ci_low,flip_rate_ci_high,hurt_rate,help_rate,b_hurt,c_helped,discordant_n,mcnemar_p,hurt_help_ratio,buried_failure_level
0,qwen/qwen3-235b-a22b-instruct-2507,target_early -> target_middle,600,0.650000,0.663333,-0.013333,-0.035000,0.008333,0.071066,0.053003,0.094669,0.031667,0.045000,19,27,46,0.301996,0.703704,mixed / borderline
1,qwen/qwen3-235b-a22b-instruct-2507,target_early -> target_late,600,0.650000,0.658333,-0.008333,-0.033333,0.016667,0.088136,0.067844,0.113756,0.043333,0.051667,26,31,57,0.596642,0.838710,mixed / borderline
2,qwen/qwen3-235b-a22b-instruct-2507,target_middle -> target_late,600,0.663333,0.658333,0.005000,-0.018333,0.028333,0.084175,0.064431,0.109263,0.045000,0.040000,27,24,51,0.779768,1.125000,mixed / borderline
